In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/email-spam-detection-dataset-classification/spam.csv
/kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin.gz
/kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin


# Imports

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [3]:
df = pd.read_csv("/kaggle/input/email-spam-detection-dataset-classification/spam.csv", encoding='ISO-8859-1')
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [4]:
filtered_df = df[df['Unnamed: 2'].notnull()]
filtered_df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAR...,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics prof...,"this wont even start........ Datz confidence..""",NaN,NaN


In [5]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


# 2. Підготовка даних:

In [7]:
df = df.rename(columns={"v1": "label", "v2": "message"})
df = df[['label', 'message']]
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [8]:
# Check the distribution of the target variable
label_distribution = df['label'].value_counts(normalize=True)
label_distribution

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

видно великий дизбаланс класів цільової змінної

## видаліть спеціальні символи, приведіть до нижнього регістру, видаліть стоп-слова тощо.

In [9]:
import spacy

In [10]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # Remove stopwords
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text


df['message'] = df['message'].apply(preprocess_text)
df.head()

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,label,message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni
2,spam,free entry wkly comp win fa cup final tkts st ...
3,ham,u dun say early hor u c already say
4,ham,nah dont think goes usf lives around though


## training and validation sets

In [11]:
df.shape

(5572, 2)

In [12]:
X_train, X_val, y_train, y_val = train_test_split(df['message'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(4457,) (1115,) (4457,) (1115,)


# 3. Перетворення текстових даних у числові вектори.

In [13]:
bow_vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 2))
tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

# Fitting vectorizers on data
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)


print(X_train_bow.shape, X_val_bow.shape, X_train_tfidf.shape, X_val_tfidf.shape)

(4457, 1000) (1115, 1000) (4457, 1000) (1115, 1000)


# 4. Використання попередньо навчених ембедингів:

In [14]:
from gensim.models import KeyedVectors

In [15]:
word2vec_path = '/kaggle/input/googlenewsvectorsnegative300/GoogleNews-vectors-negative300.bin'
word2vec_embeddings = KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

def text_to_embedding(text, embeddings, embedding_dim=300):
    words = text.split()
    vectors = [embeddings[word] for word in words if word in embeddings]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(embedding_dim)


embedding_dim = 300  # Google News Word2Vec це стандартна розмірність для цієї конкретної моделі Word2Vec 
X_train_embed = np.array([text_to_embedding(text, word2vec_embeddings, embedding_dim) for text in X_train])
X_val_embed = np.array([text_to_embedding(text, word2vec_embeddings, embedding_dim) for text in X_val])

# 5. Побудова та навчання моделей:

In [16]:
log_reg_bow = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')
rf_bow = RandomForestClassifier(n_estimators=100, random_state=42)


log_reg_tfidf = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')
rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)

log_reg_embed = LogisticRegression(max_iter=1000, random_state=42, penalty = 'l2')

In [17]:
# Train models on BoW data
log_reg_bow.fit(X_train_bow, y_train)
rf_bow.fit(X_train_bow, y_train)

# Train models on TF-IDF data
log_reg_tfidf.fit(X_train_tfidf, y_train)
rf_tfidf.fit(X_train_tfidf, y_train)

# Train models on Pretrained embeddings
log_reg_embed.fit(X_train_embed, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [18]:
# Predictions on validation data

#BoW
y_pred_log_reg_bow = log_reg_bow.predict(X_val_bow)
y_pred_rf_bow = rf_bow.predict(X_val_bow)

#TF-IDF
y_pred_log_reg_tfidf = log_reg_tfidf.predict(X_val_tfidf)
y_pred_rf_tfidf = rf_tfidf.predict(X_val_tfidf)

#Pretrained embeddings
y_pred_embed = log_reg_embed.predict(X_val_embed)

# 6. Оцінка моделей:

In [19]:
# Evaluation for each model
evaluation_results = {
    "Logistic Regression (BoW)": {
        "Accuracy": accuracy_score(y_val, y_pred_log_reg_bow),
        "AUC": roc_auc_score(y_val, log_reg_bow.predict_proba(X_val_bow)[:, 1])
    },
    "Random Forest (BoW)": {
        "Accuracy": accuracy_score(y_val, y_pred_rf_bow),
        "AUC": roc_auc_score(y_val, rf_bow.predict_proba(X_val_bow)[:, 1])
    },
    "Logistic Regression (TF-IDF)": {
        "Accuracy": accuracy_score(y_val, y_pred_log_reg_tfidf),
        "AUC": roc_auc_score(y_val, log_reg_tfidf.predict_proba(X_val_tfidf)[:, 1])
    },
    "Random Forest (TF-IDF)": {
        "Accuracy": accuracy_score(y_val, y_pred_rf_tfidf),
        "AUC": roc_auc_score(y_val, rf_tfidf.predict_proba(X_val_tfidf)[:, 1])
    },
    "Pretrained Embeddings": {
        "Accuracy": accuracy_score(y_val, y_pred_embed),
        "AUC": roc_auc_score(y_val, log_reg_embed.predict_proba(X_val_embed)[:, 1])
    }
}

evaluation_results


{'Logistic Regression (BoW)': {'Accuracy': 0.979372197309417,
  'AUC': 0.9823460752845055},
 'Random Forest (BoW)': {'Accuracy': 0.97847533632287,
  'AUC': 0.9819153223004988},
 'Logistic Regression (TF-IDF)': {'Accuracy': 0.9659192825112107,
  'AUC': 0.9814289882862979},
 'Random Forest (TF-IDF)': {'Accuracy': 0.9713004484304932,
  'AUC': 0.9819952200314034},
 'Pretrained Embeddings': {'Accuracy': 0.9506726457399103,
  'AUC': 0.9734044770519822}}

**Вибір метрик:**  
`Accuracy` - загальна точність класифікації.  
`AUC` - для визначення балансу між чутливістю та специфічністю.  
**Порівняння:**  
Моделі з BoW та TF-IDF показали вищу точність і AUC, особливо Logistic Regression (BoW), яка досягла найкращих результатів точності (97.94%) та AUC (98.28%). BoW дещо перевершує TF-IDF, оскільки краще виділяє поширені в спамі слова. Попередньо натреновані ембедінги мають нижчі показники, адже усереднені вектори втрачають важливі для спаму деталі, що робить їх менш ефективними для коротких текстів.